In [ ]:
from  langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv

# This function will load all the variables from the .env file and will 
# make them available in the os.environ dictionary (env variables)
load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

Bro API KEY Variable exists


In [6]:
from pydantic import BaseModel
from typing import Literal #literal means just defined options

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

# llm_structured_output.invoke("The movie was great")



In [ ]:
#result = llm_structured_output.invoke("The movie was great")
#result.model_dump()['movie_summary_flag']

f:\langchain_proj\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(movie_summary_flag='positive'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


'positive'

# Chain with Conditional Chains

In [7]:
# Task 1 Prompt

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "you are a movie review evaluator"),
    ("human", "please categorize the movie review as positive or negative: {input}")
])

In [8]:
# Task 2 LLM
llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

In [11]:
# Task 3 Custom Runnable
from langchain_core.runnables import RunnableLambda

# we cannot pass or run a python function directly in the chain, 
# we need to convert/wrap it in a RunnableLambda

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

# Conditional Chain 1

In [12]:
# task 1 prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a linkedin post generator"),
    ("human", "create a post for the following text for linkedin: {text}")
])


#task 2 LLM
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)


#task 3 Str Parser
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser 

# Conditional Chain 2

In [15]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [14]:
def insta_chain(text:dict):

    text = text["text"]

    # task 1 prompt
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a linkedin post generator"),
    ("human", "create a post for the following text for Instagram: {text}")])


    #task 2 LLM
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)


    #task 3 Str Parser
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser
    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)





# Final Orchestration

In [18]:
conditinal_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditinal_chain

In [19]:
final_orchestrator.invoke({"input": "I love KGF movie"})

f:\langchain_proj\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(movie_summary_flag='positive'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


'Positivity is more than a mood — it’s a strategy.\n\nWhen we choose a positive mindset at work, we improve problem-solving, strengthen resilience, and lift team morale. That doesn’t mean ignoring hard realities; it means acknowledging challenges while intentionally looking for progress, solutions, and opportunities to learn.\n\nThree simple habits that help:\n- Acknowledge small wins daily\n- Reframe setbacks as lessons\n- Give specific, timely recognition to teammates\n\nHow are you bringing positivity into your work this week? Share one small win below.  \n#positivity #leadership #mindset #teamwork #growth'

# Chain as a Runnable

In [17]:
# task 1 Beautify Function

def beautify(final_response: dict)-> dict:
    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


#task 2 final chain

#final_chain

# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")


{'linkedin': 'Just watched Pushpa: The Rise — a gritty Telugu action‑drama from director Sukumar that’s as much about survival and ambition as it is about crime.\n\nAt the center is Pushpa Raj (Allu Arjun), a downtrodden labourer who claws his way up the illicit red sandalwood‑smuggling trade in the Seshachalam forests. Using cunning, violence and sheer ambition, he builds a criminal network, clashes with rivals and law enforcement, and tries to protect his family while pursuing his love interest Srivalli (Rashmika Mandanna). The film’s pulse comes from a raw central performance, pounding music and high‑voltage action — all underscoring themes of class struggle, ambition and what people will do to survive.\n\nTakeaways for leaders and creatives: strong storytelling can humanize morally ambiguous characters; resilience and resourcefulness matter when systems are stacked against you; and craft—performance, score, direction—can elevate a familiar arc into something visceral.\n\nHave you s